In [51]:
import torch
from occhio.autoencoder import TiedLinearRelu
from occhio.distributions.hierarchical import HierarchicalSparse
from occhio.model_grid import ModelGrid, Axis
from occhio.toy_model import ToyModel
from occhio.visualization import *

device = "mps"

In [52]:
N_FEATURES, N_HIDDEN = 5, 2

axis_embed = Axis(label="p_base", values=torch.logspace(-1, 0, 6))

def create_embedding_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES, p_base=float(params["p_base"]), depth_decay=0.85, max_children=3,
            device=device, generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=0.9 ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device,
                          generator=torch.Generator(device=device).manual_seed(7)),
    )

grid_embed = ModelGrid(create_embedding_model, axes=[axis_embed])

Grouping distributions: 100%|██████████| 6/6 [00:00<00:00, 1544.39model/s]


In [6]:
grid_embed.fit(batch_size=2048, n_epochs=20_000)

Training:   4%|▎         | 737/20000 [00:03<01:39, 193.82epoch/s]


KeyboardInterrupt: 

In [5]:
print(axis_embed.values)
plot_embedding(grid_embed)


tensor([0.1000, 0.1585, 0.2512, 0.3981, 0.6310, 1.0000])


In [53]:
N_FEATURES, N_HIDDEN = 5, 2

axis_importance = Axis(label="Relative Importance", values=torch.logspace(-1, 0, 70))
axis_density    = Axis(label="p_base",     values=torch.logspace(-1, 0,  70))

def create_phase_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES, p_base=float(params["p_base"]), depth_decay=0.85, max_children=3,
            device=device, generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=float(params["Relative Importance"]) ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device,
                          generator=torch.Generator(device=device).manual_seed(7)),
    )

grid_phase = ModelGrid(create_phase_model, axes=[axis_importance, axis_density], cache_samples=True)

Grouping distributions: 100%|██████████| 4900/4900 [00:00<00:00, 7773.43model/s]


In [54]:
grid_phase.fit(batch_size=216, n_epochs=10_000)

Training:   0%|          | 31/10000 [00:02<10:58, 15.14epoch/s]


KeyboardInterrupt: 

In [58]:
# Save the models of grid_phase into a pickle
# grid_phase.save("grid_phase.pkl")

# Reinstantiate a new ModelGrid using the pickle and load()
grid_phase._models = ModelGrid.load("grid_phase.pkl")
# Optionally, verify loaded grid has the same axes, create_model, etc.

EOFError: Ran out of input

In [38]:
dist = grid_phase.models.ravel()[10].distribution
for node in dist.nodes:
    print(f"Feature {node.index}: depth={node.depth}, parent={node.parent}, children={node.children}")
print(dist.get_expected_active())

Feature 0: depth=0, parent=None, children=[1, 2, 3]
Feature 1: depth=1, parent=0, children=[4]
Feature 2: depth=1, parent=0, children=[]
Feature 3: depth=1, parent=0, children=[]
Feature 4: depth=2, parent=1, children=[]
tensor([1.0000, 0.1396, 0.1396, 0.1396, 0.0166], device='mps:0')


In [45]:
plot_phase_change_multi(grid_phase_cached, up_to=5, importance_base_axis=0)

TypeError: plot_phase_change_multi() got an unexpected keyword argument 'importance_base_axis'

In [46]:
plot_phase_change(grid_phase_cached, tracked_feature=1, importance_base_axis=0)

TypeError: plot_phase_change() got an unexpected keyword argument 'importance_base_axis'

In [ ]:
plot_phase_change(grid_phase_cached, tracked_feature=2, importance_base_axis=0)

In [ ]:
plot_phase_change(grid_phase_cached, tracked_feature=3, importance_base_axis=0)

In [ ]:
plot_phase_change(grid_phase_cached, tracked_feature=4, importance_base_axis=0)

In [18]:
# Exp 3 — Geometry: 100 features, 20 hidden (5:1), sweep p_base
# Flat importance so geometry is driven by density alone
N_FEATURES, N_HIDDEN = 5, 2

axis_geo = Axis(label="p_base", values=torch.logspace(-2, 0, 16))

def create_geometry_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES, p_base=float(params["p_base"]), depth_decay=0.85, max_children=4,
            device=device, generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=0.999 ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device,
                          generator=torch.Generator(device=device).manual_seed(7)),
    )

grid_geo = ModelGrid(create_geometry_model, axes=[axis_geo])

Grouping distributions: 100%|██████████| 16/16 [00:00<00:00, 4236.14model/s]


In [19]:
grid_geo.fit(batch_size=216, n_epochs=10_000)

Grouping distributions: 100%|██████████| 16/16 [00:00<00:00, 5122.03model/s]


In [21]:
plot_geometry(grid_geo)